In [16]:
import json
import sys

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from data.metric_cv import cv_json, quantile_quality

sys.path.insert(0, "data")

tr = pd.read_csv("data/train.csv")
te = pd.read_csv("data/test.csv")
sample = pd.read_csv("data/sample_submission.csv")
y = tr["daily_orders"].to_numpy(float)

In [17]:
df_train = tr.drop(columns=["daily_orders"])
df_test = te


def feature(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop(columns=["id"])
    df = pd.get_dummies(
        df, columns=["city", "district_type", "entrance_type"], drop_first=True
    )

    return df


In [18]:
df_train = feature(df_train)
df_test = feature(df_test)
df_test = df_test.reindex(columns=df_train.columns, fill_value=0)

In [19]:
from lightgbm import LGBMRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.experimental import enable_halving_search_cv
from sklearn.metrics import make_scorer, mean_pinball_loss
from sklearn.model_selection import HalvingRandomSearchCV

param_grid = {
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [3, 5, 7],
    "num_leaves": [7, 15, 31],
    "min_child_samples": [10, 20, 40],
}
models = {}

for q in [0.1, 0.5, 0.9]:
    neg_mean_pinball_loss_xp_scorer = make_scorer(
        mean_pinball_loss,
        alpha=q,
        greater_is_better=False,
    )
    gbr = LGBMRegressor(
        objective="quantile",
        alpha=q,
        random_state=0,
        n_jobs=1,
        verbosity=-1,
    )
    model = HalvingRandomSearchCV(
        gbr,
        param_grid,
        resource="n_estimators",
        max_resources=250,
        min_resources=50,
        scoring=neg_mean_pinball_loss_xp_scorer,
        n_jobs=2,
        random_state=0,
    ).fit(df_train, y)
    models[q] = model

print(models[0.1].best_params_)

{'num_leaves': 31, 'min_child_samples': 40, 'max_depth': 5, 'learning_rate': 0.1, 'n_estimators': 150}


In [20]:
from sklearn import clone
from sklearn.model_selection import StratifiedKFold


def predict(X):
    return np.column_stack([models[q].predict(X) for q in [0.1, 0.5, 0.9]])


splits = list(KFold(4, shuffle=True, random_state=0).split(df_train))
oof = np.zeros((len(tr), 3))

for a, b in splits:
    for j, q in enumerate([0.1, 0.5, 0.9]):
        model = clone(models[q].best_estimator_)
        model.fit(df_train.iloc[a], y[a])
        oof[b, j] = model.predict(df_train.iloc[b])
folds = [
    100 * quantile_quality(y[b], oof[b, 0], oof[b, 1], oof[b, 2], tr["city"].iloc[b])
    for _, b in splits
]

In [ ]:
df_test = df_test.reindex(columns=df_train.columns, fill_value=0)

test = np.round(predict(df_test), 1)
test = np.sort(np.clip(test, 0, 5000), axis=1)
sub = pd.DataFrame(
    {
        "id": te["id"],
        "predicted_daily_orders_p10": test[:, 0],
        "predicted_daily_orders_p50": test[:, 1],
        "predicted_daily_orders_p90": test[:, 2],
    }
)
assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample)
assert set(sub["id"]) == set(sample["id"]) and (sub.iloc[:, 1:].to_numpy() >= 0).all()
sub.to_csv("submission.csv", index=False)

cv = cv_json("cv & lightgbm", [f / 100 for f in folds])  # пишет cv.json
print(json.dumps(cv))  # последняя строка — для проверки воспроизводимости


{"name": "cv & gradboosting", "metric": "quantile_quality_x100", "folds": [72.82, 74.93, 74.79, 74.75], "cv": 74.32, "std": 0.87}
